# 04 - Prediction-level comparison of baseline and GA champion

Retrains the matched baseline under the champion's protocol, evaluates it on the same locked
hold-out, and compares the two at prediction level rather than at metric level.

Every ground-truth box is checked against each model's output at IoU >= 0.50 and confidence
>= 0.25, the threshold an operator would actually see, not the 0.001 that `val.py` uses when
integrating mAP. The comparison is restricted to the unaugmented hold-out images, because the
augmented copies could not be regenerated bit-identically across sessions.

Requires the champion's saved predictions in `champion_out/champion_eval/labels/`.
Runtime: T4 GPU, about two and a half hours.

## Session setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP  = '/content/drive/MyDrive/santos_sss.zip'
RAW  = '/content/santos_raw'
WORK = '/content/santos_yolo'
OUT  = '/content/drive/MyDrive/champion_out'
IMG, EPOCHS, SEED = 512, 100, 42

import zipfile, shutil, os
shutil.rmtree(RAW, ignore_errors=True)
zipfile.ZipFile(ZIP).extractall(RAW)
print('zip -> ok')

%cd /content
!git clone -q https://github.com/ultralytics/yolov5.git 2>/dev/null || echo 'yolov5 ok'
%cd /content/yolov5
!pip install -q -r requirements.txt
!pip install -q albumentations
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else '*** ΚΑΜΙΑ -- βαλε T4 ***')

## Collect the image/label pairs and verify the strata

In [ ]:
import os, glob, collections

IMG_EXT = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
pairs = []
for ip in glob.glob(f'{RAW}/**/*', recursive=True):
    if os.path.splitext(ip)[1].lower() not in IMG_EXT:
        continue
    lp = os.path.splitext(ip)[0] + '.txt'
    if os.path.exists(lp):
        pairs.append((os.path.splitext(os.path.basename(ip))[0], ip, lp))
pairs.sort(key=lambda r: r[0])

def read_boxes(lp):
    out=[]
    for line in open(lp):
        p=line.split()
        if len(p)>=5: out.append([int(float(p[0])), *map(float,p[1:5])])
    return out

def stratum(b):
    cs={x[0] for x in b}
    return 'bg' if not cs else 'milco' if cs=={0} else 'nombo' if cs=={1} else 'both'

boxes_of={s:read_boxes(l) for s,i,l in pairs}
strata  ={s:stratum(b) for s,b in boxes_of.items()}
by=collections.Counter(strata.values())
EXP={'bg':866,'milco':181,'both':74,'nombo':49}
print(f'ζευγη: {len(pairs)} (αναμ. 1170)')
for k,v in EXP.items():
    print(f'  {k:6s}: {by[k]:5d} / {v:<5d} {"OK" if by[k]==v else "<<<"}')
print('\n' + ('ΤΑΙΡΙΑΖΕΙ' if dict(by)==EXP else '*** ΔΙΑΦΟΡΑ -- στειλε την εξοδο ***'))

## Offline geometric augmentation, seed 42

In [ ]:
import albumentations as A, cv2, numpy as np, random, shutil
random.seed(SEED); np.random.seed(SEED)
MIN_SIDE = 1e-4

def sanitise(cls, bxs):
    oc, ob = [], []
    for c_, (cx, cy, w, h) in zip(cls, bxs):
        x0,y0,x1,y1 = cx-w/2, cy-h/2, cx+w/2, cy+h/2
        x0,y0 = max(x0,0.0), max(y0,0.0)
        x1,y1 = min(x1,1.0), min(y1,1.0)
        if x1-x0 < MIN_SIDE or y1-y0 < MIN_SIDE: continue
        oc.append(c_); ob.append([(x0+x1)/2,(y0+y1)/2,x1-x0,y1-y0])
    return oc, ob

clean = {}
for stem, ip, lp in pairs:
    bb = boxes_of[stem]
    clean[stem] = sanitise([b[0] for b in bb], [b[1:] for b in bb])

def build_tf():
    aff = dict(rotate=(-10,10), translate_percent=(-0.1,0.1), scale=(0.8,1.2),
               border_mode=cv2.BORDER_CONSTANT, p=0.9)
    for kw in ('fill','cval'):
        try:
            return A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2),
                              A.Affine(**{**aff, kw: 0})],
                   bbox_params=A.BboxParams(format='yolo', label_fields=['cls'],
                                            min_visibility=0.3))
        except TypeError: pass
    raise RuntimeError('API Albumentations')
tf = build_tf(); print('albumentations', A.__version__)

AUG = f'{WORK}/_all'
shutil.rmtree(WORK, ignore_errors=True); os.makedirs(AUG)

def wl(path, cls, bxs):
    oc, ob = sanitise(cls, bxs)
    with open(path,'w') as f:
        for c_, b in zip(oc, ob):
            f.write(f'{int(c_)} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}\n')

groups = {}
for idx,(stem, ip, lp) in enumerate(pairs):
    img = cv2.imread(ip)
    cls, bxs = clean[stem]
    cv2.imwrite(f'{AUG}/{stem}.jpg', img); wl(f'{AUG}/{stem}.txt', cls, bxs)
    members=[stem]
    for k in range(3):
        r = {'image':img,'bboxes':bxs,'cls':cls}
        for _ in range(5):
            r = tf(image=img, bboxes=bxs, cls=cls)
            if not cls or len(r['bboxes'])>0: break
        nm=f'{stem}_aug{k}'
        cv2.imwrite(f'{AUG}/{nm}.jpg', r['image']); wl(f'{AUG}/{nm}.txt', r['cls'], r['bboxes'])
        members.append(nm)
    groups[stem]=members
    if (idx+1)%300==0: print(f'  {idx+1}/{len(pairs)}')
print(f'\n{len(os.listdir(AUG))//2} εικονες / {len(groups)} ομαδες (αναμ. 4680 / 1170)')

bad = 0
checked = 0
for stem in list(groups)[:400]:
    for m in groups[stem]:
        p = f'{AUG}/{m}.txt'
        txt = open(p).read()
        if '\\n' in txt:
            bad += 1; continue
        for line in txt.strip().splitlines():
            fl = line.split()
            if len(fl) != 5: bad += 1; break
            try: [float(v) for v in fl]
            except ValueError: bad += 1; break
        checked += 1
print(f'ελεγχθηκαν {checked} labels, κατεστραμμενα: {bad}')
assert bad == 0, 'ΚΑΤΕΣΤΡΑΜΜΕΝΑ LABELS -- ΜΗΝ ΠΡΟΧΩΡΗΣΕΙΣ'
print('τα labels ειναι εγκυρα')

## Split and the baseline hyperparameters

In [ ]:
import random, yaml
rng = random.Random(SEED)
bys = {}
for s in groups: bys.setdefault(strata[s], []).append(s)
hold, pool = [], []
for k in sorted(bys):
    g = sorted(bys[k]); rng.shuffle(g)
    nh = round(len(g)*0.10)
    hold += g[:nh]; pool += g[nh:]
print(f'hold-out {len(hold)} ομαδες = {len(hold)*4} αρχεια (αναμ. 117 / 468)')
print(f'pool     {len(pool)} ομαδες = {len(pool)*4} αρχεια (αναμ. 1053 / 4212)')

def mat(split, stems):
    di, dl = f'{WORK}/images/{split}', f'{WORK}/labels/{split}'
    os.makedirs(di, exist_ok=True); os.makedirs(dl, exist_ok=True)
    for s in stems:
        for m in groups[s]:
            shutil.copy(f'{AUG}/{m}.jpg', f'{di}/{m}.jpg')
            shutil.copy(f'{AUG}/{m}.txt', f'{dl}/{m}.txt')
    return len(os.listdir(di))

n_tr = mat('train', pool); n_te = mat('test', hold)
shutil.copytree(f'{WORK}/images/train', f'{WORK}/images/val')
shutil.copytree(f'{WORK}/labels/train', f'{WORK}/labels/val')
print(f'train {n_tr} | test {n_te}')

yaml.safe_dump({'path':WORK,'train':'images/train','val':'images/val',
                'test':'images/test','nc':2,'names':['MILCO','NOMBO']},
               open('/content/yolov5/data/santos.yaml','w'), sort_keys=False)

hyp_base = {'lr0':0.01,'lrf':0.01,'momentum':0.937,'weight_decay':0.0005,
            'box':0.05,'cls':0.5,'obj':1.0,
            'warmup_epochs':3.0,'warmup_momentum':0.8,'warmup_bias_lr':0.1,
            'cls_pw':1.0,'obj_pw':1.0,'iou_t':0.20,'anchor_t':4.0,'fl_gamma':0.0,
            'hsv_h':0.0,'hsv_s':0.0,'hsv_v':0.0,'degrees':0.0,'translate':0.0,
            'scale':0.0,'shear':0.0,'perspective':0.0,'flipud':0.0,'fliplr':0.0,
            'mosaic':0.0,'mixup':0.0,'copy_paste':0.0}
yaml.safe_dump(hyp_base, open('/content/yolov5/data/hyps/hyp_baseline.yaml','w'),
               sort_keys=False)
print('\\nhyp_baseline.yaml: lr0=0.01 obj=1.0 cls=0.5  (defaults)')

## Train the matched baseline

In [ ]:
cmd = (f"python train.py --img {IMG} --epochs {EPOCHS} --batch 16 "
       f"--data data/santos.yaml --hyp data/hyps/hyp_baseline.yaml "
       f"--weights yolov5s.pt --cache "
       f"--project {OUT} --name baseline --exist-ok")
print(cmd)
get_ipython().system(cmd)

## Evaluate the baseline on the same locked hold-out

In [ ]:
ch_lbl = f'{OUT}/champion_eval/labels'
assert os.path.isdir(ch_lbl), (
    'ΔΕΝ ΒΡΕΘΗΚΑΝ οι προβλεψεις του champion στο ' + ch_lbl +
    ' -- χωρις αυτες δεν γινεται συγκριση.')
print('champion labels:', len(os.listdir(ch_lbl)), 'αρχεια')

cmd = (f"python val.py --img {IMG} --batch 16 --data data/santos.yaml "
       f"--weights {OUT}/baseline/weights/last.pt "
       f"--task test --verbose --save-txt --save-conf "
       f"--project {OUT} --name baseline_eval --exist-ok "
       f"2>&1 | tee {OUT}/baseline_eval_log.txt")
get_ipython().system(cmd)

## Prediction-level contingency table

The full two-by-two of which model recovered which contact, split between original and
augmented hold-out images, with the confidence distributions and the candidate counts.

In [ ]:
import os, glob, collections, numpy as np
CONF, IOU_T = 0.25, 0.50
base_lbl = f'{OUT}/baseline_eval/labels'
ga_lbl   = f'{OUT}/champion_eval/labels'

def load_pred(d, stem, conf=CONF):
    p = f'{d}/{stem}.txt'; out = []
    if os.path.exists(p):
        for line in open(p):
            f = line.split()
            if len(f) >= 6 and float(f[5]) >= conf:
                out.append((int(f[0]), *map(float, f[1:5]), float(f[5])))
    return out

def to_xyxy(b):
    cx, cy, w, h = b[:4]
    return cx-w/2, cy-h/2, cx+w/2, cy+h/2

def iou(a, b):
    ax0,ay0,ax1,ay1 = to_xyxy(a); bx0,by0,bx1,by1 = to_xyxy(b)
    ix0,iy0 = max(ax0,bx0), max(ay0,by0)
    ix1,iy1 = min(ax1,bx1), min(ay1,by1)
    iw, ih = max(0.0, ix1-ix0), max(0.0, iy1-iy0)
    inter = iw*ih
    ua = (ax1-ax0)*(ay1-ay0) + (bx1-bx0)*(by1-by0) - inter
    return inter/ua if ua > 0 else 0.0

def hit(gt, preds):
    best = None
    for p in preds:
        if p[0] == gt[0] and iou(gt[1:5], p[1:5]) >= IOU_T:
            if best is None or p[5] > best[5]: best = p
    return best

CELLS = {}
tally = collections.Counter()
nfile = collections.Counter()

for f in sorted(glob.glob(f'{WORK}/labels/test/*.txt')):
    stem = os.path.splitext(os.path.basename(f))[0]
    t = 'orig' if '_aug' not in stem else 'aug'
    nfile[(t,'files')] += 1
    nfile[(t,'pb')] += os.path.exists(f'{base_lbl}/{stem}.txt')
    nfile[(t,'pg')] += os.path.exists(f'{ga_lbl}/{stem}.txt')
    gts = [[int(float(x.split()[0]))] + [float(v) for v in x.split()[1:5]]
           for x in open(f) if len(x.split()) >= 5]
    if not gts: continue
    pb, pg = load_pred(base_lbl, stem), load_pred(ga_lbl, stem)
    for gt in gts:
        hb, hg = hit(gt, pb), hit(gt, pg)
        key = ('ga_only'   if (hb is None and hg is not None) else
               'base_only' if (hb is not None and hg is None) else
               'both_hit'  if (hb is not None) else 'both_miss')
        tally[(t,key)] += 1
        CELLS.setdefault((t,key), []).append((stem, gt, hb, hg))

print('ΠΙΝΑΚΑΣ 2x2  (IoU>=0.50, conf>=0.25)')
print(f"{'':12s}{'ΠΡΩΤΟΤΥΠΕΣ':>12s}{'ΕΠΑΥΞΗΜΕΝΕΣ':>13s}{'ΣΥΝΟΛΟ':>9s}")
for k in ('both_hit','ga_only','base_only','both_miss'):
    a, b = tally[('orig',k)], tally[('aug',k)]
    print(f'{k:12s}{a:>12d}{b:>13d}{a+b:>9d}')
a = sum(tally[('orig',k)] for k in ('both_hit','ga_only','base_only','both_miss'))
b = sum(tally[('aug',k)]  for k in ('both_hit','ga_only','base_only','both_miss'))
print(f"{'ΣΥΝΟΛΟ':12s}{a:>12d}{b:>13d}{a+b:>9d}")

print('\nΑΡΧΕΙΑ ΜΕ ΕΣΤΩ ΜΙΑ ΠΡΟΒΛΕΨΗ (conf>=0.001)')
for t in ('orig','aug'):
    print(f'  {t}: {nfile[(t,"files")]} εικονες | '
          f'baseline {nfile[(t,"pb")]} | GA {nfile[(t,"pg")]}')

print('\nΚΑΤΑΝΟΜΗ CONFIDENCE -- μονο ΠΡΩΤΟΤΥΠΕΣ')
for name, d in (('baseline', base_lbl), ('GA', ga_lbl)):
    cs = []
    for f in glob.glob(f'{WORK}/labels/test/*.txt'):
        stem = os.path.splitext(os.path.basename(f))[0]
        if '_aug' in stem: continue
        p = f'{d}/{stem}.txt'
        if os.path.exists(p):
            for line in open(p):
                q = line.split()
                if len(q) >= 6: cs.append(float(q[5]))
    cs = np.array(cs) if cs else np.array([0.0])
    print(f'  {name:9s} n={len(cs):5d}  >=0.10:{(cs>=.10).sum():5d}'
          f'  >=0.25:{(cs>=.25).sum():5d}  >=0.50:{(cs>=.50).sum():5d}'
          f'  max={cs.max():.2f}')

print('\nΥΠΟΨΗΦΙΑ -- ΜΟΝΟ ΠΡΩΤΟΤΥΠΕΣ ΕΙΚΟΝΕΣ')
for k in ('ga_only','base_only','both_miss'):
    v = CELLS.get(('orig',k), [])
    if k == 'ga_only':   v = sorted(v, key=lambda r: -r[3][5])
    if k == 'base_only': v = sorted(v, key=lambda r: -r[2][5])
    print(f'  --- {k} ({len(v)}) ---')
    for stem, gt, hb, hg in v[:6]:
        cls = ['MILCO','NOMBO'][gt[0]]
        print(f'    {stem:16s} {cls:6s} area={gt[3]*gt[4]*100:.3f}%  '
              f'baseline={"—" if hb is None else f"{hb[5]:.2f}"}  '
              f'GA={"—" if hg is None else f"{hg[5]:.2f}"}')

## Figure 4.6

Four rows by three columns: ground truth, baseline, GA champion. Each row is selected
automatically from one population of the table above, so the figure is not a showcase:
(a) recovered only by the champion, (b) a baseline box with the wrong class label,
(c) recovered only by the baseline, (d) missed by both.

In [ ]:
import os, glob, zipfile, shutil, random, numpy as np
from PIL import Image, ImageDraw, ImageFont

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

ZIP='/content/drive/MyDrive/santos_sss.zip'; RAW='/content/santos_raw'
OUT='/content/drive/MyDrive/champion_out'; SEED=42
CONF, IOU_T = 0.25, 0.50
base_lbl=f'{OUT}/baseline_eval/labels'; ga_lbl=f'{OUT}/champion_eval/labels'
for p in (base_lbl, ga_lbl):
    assert os.path.isdir(p), 'ΛΕΙΠΕΙ: '+p
if not os.path.isdir(RAW):
    zipfile.ZipFile(ZIP).extractall(RAW)

IMG_EXT={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
pairs=[]
for ip in glob.glob(f'{RAW}/**/*', recursive=True):
    if os.path.splitext(ip)[1].lower() not in IMG_EXT: continue
    lp=os.path.splitext(ip)[0]+'.txt'
    if os.path.exists(lp): pairs.append((os.path.splitext(os.path.basename(ip))[0],ip,lp))
pairs.sort(key=lambda r:r[0])
def read_boxes(lp):
    o=[]
    for line in open(lp):
        p=line.split()
        if len(p)>=5: o.append([int(float(p[0])),*map(float,p[1:5])])
    return o
def stratum(b):
    cs={x[0] for x in b}
    return 'bg' if not cs else 'milco' if cs=={0} else 'nombo' if cs=={1} else 'both'
boxes_of={s:read_boxes(l) for s,i,l in pairs}
strata={s:stratum(b) for s,b in boxes_of.items()}
IMGPATH={s:i for s,i,l in pairs}
rng=random.Random(SEED); bys={}
for s,_,_ in pairs: bys.setdefault(strata[s],[]).append(s)
hold=[]
for k in sorted(bys):
    g=sorted(bys[k]); rng.shuffle(g); hold+=g[:round(len(g)*0.10)]
assert len(hold)==117, 'Ο ΧΩΡΙΣΜΟΣ ΔΕΝ ΑΝΑΠΑΡΗΧΘΗ'

MIN_SIDE=1e-4
def sanitise(bb):
    o=[]
    for c,cx,cy,w,h in bb:
        x0,y0,x1,y1=max(cx-w/2,0.),max(cy-h/2,0.),min(cx+w/2,1.),min(cy+h/2,1.)
        if x1-x0<MIN_SIDE or y1-y0<MIN_SIDE: continue
        o.append([c,(x0+x1)/2,(y0+y1)/2,x1-x0,y1-y0])
    return o
def load_pred(d,stem,conf=CONF):
    p=f'{d}/{stem}.txt'; o=[]
    if os.path.exists(p):
        for line in open(p):
            f=line.split()
            if len(f)>=6 and float(f[5])>=conf:
                o.append((int(f[0]),*map(float,f[1:5]),float(f[5])))
    return o
def to_xyxy(b):
    cx,cy,w,h=b[:4]; return cx-w/2,cy-h/2,cx+w/2,cy+h/2
def iou(a,b):
    ax0,ay0,ax1,ay1=to_xyxy(a); bx0,by0,bx1,by1=to_xyxy(b)
    iw=max(0.,min(ax1,bx1)-max(ax0,bx0)); ih=max(0.,min(ay1,by1)-max(ay0,by0))
    inter=iw*ih; ua=(ax1-ax0)*(ay1-ay0)+(bx1-bx0)*(by1-by0)-inter
    return inter/ua if ua>0 else 0.
def hit(gt,preds):
    best=None
    for p in preds:
        if p[0]==gt[0] and iou(gt[1:5],p[1:5])>=IOU_T:
            if best is None or p[5]>best[5]: best=p
    return best

areas={0:[],1:[]}
for s,_,_ in pairs:
    for c,cx,cy,w,h in sanitise(boxes_of[s]): areas[c].append(w*h)
MEAN={c:(np.mean(v) if v else 0.) for c,v in areas.items()}

CN=['MILCO','NOMBO']
C={'ga_only':[], 'base_only':[], 'both_miss':[]}
for stem in sorted(hold):
    gts=sanitise(boxes_of[stem])
    if not gts: continue
    pb,pg=load_pred(base_lbl,stem),load_pred(ga_lbl,stem)
    for gt in gts:
        hb,hg=hit(gt,pb),hit(gt,pg)
        if hb is None and hg is not None: C['ga_only'].append((stem,gt,hb,hg,pb,pg))
        elif hb is not None and hg is None: C['base_only'].append((stem,gt,hb,hg,pb,pg))
        elif hb is None and hg is None: C['both_miss'].append((stem,gt,hb,hg,pb,pg))

ga_milco=[r for r in C['ga_only'] if r[1][0]==0]
rA=max(ga_milco or C['ga_only'], key=lambda r: r[3][5])
wrong=[]
for rec in C['ga_only']:
    for p in rec[4]:
        if p[0]!=rec[1][0] and iou(rec[1][1:5],p[1:5])>=IOU_T: wrong.append((rec,p)); break
rB=max(wrong,key=lambda t:t[1][5])[0] if wrong else None
rC=max(C['base_only'], key=lambda r: r[2][5])
rD=min(C['both_miss'], key=lambda r: abs(r[1][3]*r[1][4]-MEAN[r[1][0]]))
chosen=[rA]+([rB] if rB is not None and rB[0]!=rA[0] else [])+[rC,rD]
labs=[('(a)','recovered only by the\nGA-optimised model'),
      ('(b)','baseline box carries the\nwrong class label'),
      ('(c)','recovered only by the\nbaseline'),
      ('(d)','missed by both; area near\nthe class mean')]
if len(chosen)==3: labs=[labs[0],labs[2],labs[3]]

print(f'ga_only {len(C["ga_only"])} | base_only {len(C["base_only"])} | '
      f'both_miss {len(C["both_miss"])} | wrong-class {len(wrong)}')
print(f'ΣΕΙΡΕΣ ΣΤΟ ΣΧΗΜΑ: {len(chosen)}')
for i,(stem,gt,hb,hg,pb,pg) in enumerate(chosen):
    print(f'  {labs[i][0]} {stem:14s} GT={CN[gt[0]]:6s} area={gt[3]*gt[4]*100:.3f}%  '
          f'baseline={"—" if hb is None else f"{hb[5]:.2f}"}  '
          f'GA={"—" if hg is None else f"{hg[5]:.2f}"}')

GT_C=(46,139,87); BASE_C=(193,101,26); GA_C=(45,105,160)
TILE=380
F_HDR, F_ROW, F_TXT = 40, 40, 30
try:
    B='/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
    R='/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
    F  = ImageFont.truetype(B, F_HDR)
    FS = ImageFont.truetype(B, F_TXT)
    FT = ImageFont.truetype(R, F_TXT)
    FB = ImageFont.truetype(B, F_TXT)
except Exception:
    F=FS=FT=FB=ImageFont.load_default()

def panel(stem, gt, matched, preds, colour, kind):
    im=Image.open(IMGPATH[stem]).convert('RGB'); W,H=im.size
    side=max(gt[3]*W, gt[4]*H); win=int(min(min(W,H), max(160, 6*side)))
    cx,cy=gt[1]*W, gt[2]*H
    x0=int(max(0,min(W-win, cx-win/2))); y0=int(max(0,min(H-win, cy-win/2)))
    tile=im.crop((x0,y0,x0+win,y0+win)).resize((TILE,TILE), Image.LANCZOS)
    d=ImageDraw.Draw(tile); k=TILE/win

    def plate(xy, txt, font, fg):
        l,t,r,b = d.textbbox(xy, txt, font=font)
        d.rectangle([l-6,t-4,r+6,b+4], fill=(15,15,15))
        d.text(xy, txt, font=font, fill=fg)

    def box(b,col,wd,txt=None):
        bx0=(b[1]*W-b[3]*W/2-x0)*k; by0=(b[2]*H-b[4]*H/2-y0)*k
        bx1=(b[1]*W+b[3]*W/2-x0)*k; by1=(b[2]*H+b[4]*H/2-y0)*k
        d.rectangle([bx0,by0,bx1,by1], outline=col, width=wd)
        if txt: plate((bx0, max(2, by0-F_TXT-10)), txt, FB, col)

    box(gt, GT_C, 4, CN[gt[0]] if kind=='gt' else None)
    info=[]
    if kind!='gt':
        for p in preds:
            pcx,pcy=p[1]*W, p[2]*H
            if not (x0<=pcx<=x0+win and y0<=pcy<=y0+win): continue
            v=iou(gt[1:5],p[1:5])
            same = matched is not None and p==matched
            box(p, colour, 6 if same else 3, f'{CN[p[0]]} {p[5]:.2f}')
            info.append((CN[p[0]], p[5], v, same))
        if matched is None:
            if not info: msg='no box above 0.25'
            elif any(v>=IOU_T and c!=CN[gt[0]] for c,_,v,_ in info): msg='wrong class'
            else: msg=f'best IoU {max(v for _,_,v,_ in info):.2f} < 0.50'
            plate((10, TILE-F_TXT-14), msg, FS, (255,80,80))
    return tile, info

rows=[]; REPORT=[]
for stem,gt,hb,hg,pb,pg in chosen:
    t0,_ =panel(stem,gt,None,[],GT_C,'gt')
    t1,i1=panel(stem,gt,hb,pb,BASE_C,'p')
    t2,i2=panel(stem,gt,hg,pg,GA_C,'p')
    rows.append([t0,t1,t2]); REPORT.append((stem,gt,i1,i2))

PAD,HDR,LBL=14,58,250; R_=len(rows)
Wf=LBL+3*TILE+4*PAD; Hf=HDR+R_*TILE+(R_+1)*PAD
fig=Image.new('RGB',(Wf,Hf),(255,255,255)); d=ImageDraw.Draw(fig)
for j_,t in enumerate(['Ground truth','Baseline (YOLOv5 defaults)','GA-optimised champion']):
    w = d.textbbox((0,0), t, font=F)[2]
    d.text((LBL+PAD+j_*(TILE+PAD)+TILE//2-w//2, 10), t, font=F, fill=(0,0,0))
for i,row in enumerate(rows):
    y=HDR+PAD+i*(TILE+PAD)
    d.text((14, y+TILE//2-64), labs[i][0], font=F, fill=(0,0,0))
    d.multiline_text((14, y+TILE//2-14), labs[i][1], font=FT, fill=(70,70,70), spacing=8)
    for j_,t in enumerate(row): fig.paste(t,(LBL+PAD+j_*(TILE+PAD), y))
fig.save('/content/fig_baseline_vs_ga.png', quality=95)
shutil.copy('/content/fig_baseline_vs_ga.png', f'{OUT}/fig_baseline_vs_ga.png')

print('\n=== ΤΙ ΖΩΓΡΑΦΙΣΤΗΚΕ ===')
for r,(stem,gt,i1,i2) in enumerate(REPORT):
    print(f'\n{labs[r][0]} {stem}  GT={CN[gt[0]]}')
    for nm,inf in (('baseline',i1),('GA      ',i2)):
        if not inf: print(f'   {nm}: καμια προβλεψη >=0.25 στο παραθυρο')
        for c,conf,v,same in inf:
            print(f'   {nm}: {c:6s} conf={conf:.2f} IoU={v:.2f}{"   <-- ΜΕΤΡΑΕΙ" if same else ""}')
print('\nγραφτηκε:', fig.size)
from google.colab import files; files.download('/content/fig_baseline_vs_ga.png')
fig